# 🧠 AI 3D Stress Validator — V2: GNN Edition

> **Physics-Informed Structural Validation** using Graph Neural Networks.
>
> Instead of voxelizing meshes into 3D grids, V2 converts STL files into **mesh-graphs** where every vertex is a node and every edge carries structural information. A **Graph Convolutional Network** then "simulates" stress flow through the geometry.

| V1 (Voxel/CNN) | V2 (Graph/GNN) |
|---|---|
| Binary occupancy grid | Vertex positions + surface normals |
| Grid scans empty space | Graph follows real geometry |
| No edge info | Euclidean distance on every edge |
| 3D CNN (Conv3D) | GCN with message passing |

---

**Runtime:** Set to **GPU** → `Runtime → Change runtime type → T4 GPU`

## Phase 0 — Setup & Dependencies

In [ ]:
# 0.1  Install dependencies
# PyTorch Geometric requires matching torch + CUDA versions
import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'

print(f'PyTorch {TORCH_VERSION}, CUDA {CUDA_VERSION}')

!pip install -q torch-geometric
!pip install -q trimesh open3d scipy pandas matplotlib scikit-learn

# Verify installation
import torch_geometric
print(f'✅ PyTorch Geometric {torch_geometric.__version__} installed.')

In [ ]:
# 0.2  Mount Google Drive (for data & model saving)
from google.colab import drive
drive.mount('/content/drive')

import os

# --- Paths on Google Drive ---
# DATA_ROOT: folder with your 250 STL files (same folder used in V1)
DATA_ROOT        = '/content/drive/MyDrive/AI_Stress_Validator'
DRIVE_MODEL_DIR  = '/content/drive/MyDrive/AI_Stress_Validator/GNN_Model'
GRAPH_CACHE_DIR  = '/content/drive/MyDrive/AI_Stress_Validator/Graph_Cache'

os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
os.makedirs(GRAPH_CACHE_DIR, exist_ok=True)

print(f'📂 Data root     : {DATA_ROOT}')
print(f'📂 Model save dir: {DRIVE_MODEL_DIR}')
print(f'📂 Graph cache   : {GRAPH_CACHE_DIR}')

In [ ]:
# 0.3  Configuration constants
NUM_SAMPLES  = 250                         # Number of STL samples to use

# --- Preprocessing ---
TARGET_FACES = 2000                        # QEM decimation target (~80% reduction)

# --- Labels ---
STRESS_FAIL_THRESHOLD = 0.80               # quantile for pass/fail split

# --- GNN Architecture ---
IN_CHANNELS      = 6                       # [x, y, z, nx, ny, nz]
HIDDEN_CHANNELS  = 128                     # latent dimension
NUM_GNN_LAYERS   = 4                       # message-passing depth
EDGE_DIM         = 1                       # [distance]
DROPOUT          = 0.3

# --- Training ---
BATCH_SIZE   = 16
LEARNING_RATE = 1e-3
NUM_EPOCHS   = 80
WEIGHT_DECAY = 1e-4
VAL_SPLIT    = 0.2

# --- Loss weights ---
LAMBDA_STRESS = 1.0   # weight for stress regression loss (MSE)
LAMBDA_RISK   = 0.5   # weight for node-risk classification loss (BCE)

# --- Device ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 0.4  Clone repository (for utils/)
REPO_URL = 'https://github.com/janu3605/AI_3D_Tolerance_Stress_Validator.git'
REPO_DIR = '/content/AI_3D_Tolerance_Stress_Validator'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already cloned at {REPO_DIR}')

# Add to Python path
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('✅ Repository ready.')

## Phase 1 — Data Loading & Graph Preprocessing

We load the 250 STL files from Google Drive, read **all** available stress 
columns from `bracket_labels.csv`, then convert each mesh to a graph using 
QEM decimation + surface normals + edge distance injection.

In [ ]:
# 1.1  Discover STL files on Google Drive
import glob

stl_files = sorted(glob.glob(os.path.join(DATA_ROOT, '**', '*.stl'), recursive=True))
print(f'Found {len(stl_files)} STL files in Drive.')
stl_files = stl_files[:NUM_SAMPLES]
print(f'Using {len(stl_files)} samples.')

if len(stl_files) == 0:
    print('\n⚠️  No STL files found! Make sure your 250 STL files are in:')
    print(f'    {DATA_ROOT}/')
    print('   (or a subfolder within it)')

In [ ]:
# 1.2  Load stress labels from bracket_labels.csv (ALL stress columns)
import pandas as pd
import numpy as np

# Search for the CSV on Google Drive
csv_paths = (
    glob.glob(os.path.join(DATA_ROOT, '**', 'bracket_labels.csv'), recursive=True) +
    glob.glob('/content/drive/MyDrive/**/bracket_labels.csv', recursive=True)
)

if not csv_paths:
    raise FileNotFoundError(
        'bracket_labels.csv not found! Upload it to '
        f'{DATA_ROOT}/ or your Google Drive.'
    )

csv_path = csv_paths[0]
print(f'Loading FEA data from: {csv_path}')
df = pd.read_csv(csv_path)
print(f'Loaded {len(df)} rows, {len(df.columns)} columns.')
print(f'\nAll columns: {list(df.columns)}')

# --- Auto-discover ALL stress-related columns ---
# Instead of hardcoding only 4 columns, we dynamically find every
# column that contains 'stress' in its name. This captures all
# FEA load cases (vertical, horizontal, diagonal, torsional, and
# any others present in the dataset).
STRESS_COLS = [c for c in df.columns if 'stress' in c.lower()]

if not STRESS_COLS:
    raise ValueError(
        'No stress columns found in bracket_labels.csv! '
        f'Available columns: {list(df.columns)}'
    )

print(f'\n🔬 Found {len(STRESS_COLS)} stress columns:')
for col in STRESS_COLS:
    print(f'   • {col}  (range: {df[col].min():.2f} — {df[col].max():.2f} MPa)')

# Compute the worst-case (max) stress across ALL load cases per part
df['max_stress_all'] = df[STRESS_COLS].max(axis=1)

# --- Labeling strategy ---
# Parts above the threshold quantile are labeled FAIL (1)
threshold = df['max_stress_all'].quantile(STRESS_FAIL_THRESHOLD)
df['label'] = (df['max_stress_all'] >= threshold).astype(float)

# Build lookups
stress_lookup = {str(row['item_name']): float(row['max_stress_all']) for _, row in df.iterrows()}
label_lookup  = {str(row['item_name']): float(row['label']) for _, row in df.iterrows()}

# Summary
n_fail = int(df['label'].sum())
print(f'\n📊 Stress Statistics:')
print(f'   Min stress  : {df["max_stress_all"].min():.2f} MPa')
print(f'   Mean stress : {df["max_stress_all"].mean():.2f} MPa')
print(f'   Median      : {df["max_stress_all"].median():.2f} MPa')
print(f'   Max stress  : {df["max_stress_all"].max():.2f} MPa')
print(f'   Fail threshold (q={STRESS_FAIL_THRESHOLD}): {threshold:.2f} MPa')
print(f'\n🏷️  Labels: {len(df)-n_fail} PASS, {n_fail} FAIL ({100*n_fail/len(df):.1f}% fail rate)')

In [ ]:
# 1.3  Convert STL files to graphs (the key V2 step!)
#
# This cell checks for a cached version on Drive first.
# If graphs were already processed, it loads them instantly.
# Otherwise, it runs the full conversion and saves to Drive.

from utils.mesh_to_graph import mesh_to_graph
import time
import pickle

GRAPH_CACHE_FILE = os.path.join(GRAPH_CACHE_DIR, f'graphs_f{TARGET_FACES}_n{NUM_SAMPLES}.pkl')

if os.path.exists(GRAPH_CACHE_FILE):
    print(f'📦 Loading cached graphs from Drive: {GRAPH_CACHE_FILE}')
    with open(GRAPH_CACHE_FILE, 'rb') as f:
        graph_list = pickle.load(f)
    print(f'✅ Loaded {len(graph_list)} cached graphs.')
else:
    print(f'🔄 Converting {len(stl_files)} STL files to graphs...')
    print(f'   Target faces after decimation: {TARGET_FACES}')
    print(f'   Node features: [x, y, z, nx, ny, nz] (6D)')
    print(f'   Edge features: [euclidean_distance] (1D)')
    print()

    start_time = time.time()
    graph_list = []
    skipped = 0

    for i, stl_path in enumerate(stl_files):
        try:
            # Convert mesh to graph
            data = mesh_to_graph(stl_path, target_faces=TARGET_FACES)

            # Attach labels
            basename = os.path.splitext(os.path.basename(stl_path))[0]
            stress_val = stress_lookup.get(basename, 0.0)
            label = label_lookup.get(basename, 0.0)

            data.y = torch.tensor([label], dtype=torch.float32)
            data.stress_value = torch.tensor([stress_val], dtype=torch.float32)
            data.item_name = basename

            graph_list.append(data)

            if (i + 1) % 25 == 0:
                elapsed = time.time() - start_time
                rate = (i + 1) / elapsed
                print(f'  ✅ {i+1}/{len(stl_files)} done  '
                      f'({rate:.1f} meshes/sec, '
                      f'last graph: {data.num_nodes} nodes, '
                      f'{data.edge_index.shape[1]} edges)')

        except Exception as e:
            skipped += 1
            print(f'  ⚠️  Skipped {os.path.basename(stl_path)}: {e}')

    elapsed = time.time() - start_time
    print(f'\n✅ Converted {len(graph_list)} graphs in {elapsed:.1f}s ({skipped} skipped)')

    # Cache to Drive for next time
    print(f'💾 Saving graph cache to Drive: {GRAPH_CACHE_FILE}')
    with open(GRAPH_CACHE_FILE, 'wb') as f:
        pickle.dump(graph_list, f)
    print('   Saved!')

In [ ]:
# 1.4  Inspect a sample graph
sample = graph_list[0]
print('📐 Sample graph structure:')
print(f'   Nodes     : {sample.num_nodes}')
print(f'   Edges     : {sample.edge_index.shape[1]} (bidirectional)')
print(f'   Node feat : {sample.x.shape}  →  [x, y, z, nx, ny, nz]')
print(f'   Edge attr : {sample.edge_attr.shape}  →  [distance]')
print(f'   Label     : {sample.y.item()} ({"FAIL" if sample.y.item() == 1 else "PASS"})')
print(f'   Stress    : {sample.stress_value.item():.2f} MPa')
print(f'   Name      : {sample.item_name}')

In [ ]:
# 1.5  Visualize a sample mesh-graph
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 5))

# --- Plot 1: Node positions colored by normal direction ---
ax1 = fig.add_subplot(131, projection='3d')
pos = sample.pos.numpy()
normals = sample.x[:, 3:6].numpy()
# Color by normal z-component (highlights flat vs curved surfaces)
colors = (normals[:, 2] + 1) / 2  # map [-1,1] to [0,1]
sc = ax1.scatter(pos[:, 0], pos[:, 1], pos[:, 2],
                 c=colors, cmap='coolwarm', s=3, alpha=0.7)
ax1.set_title(f'Nodes ({sample.num_nodes})', fontsize=10)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# --- Plot 2: Edge connections (subset for clarity) ---
ax2 = fig.add_subplot(132, projection='3d')
edge_idx = sample.edge_index.numpy()
# Plot every Nth edge to avoid visual clutter
step = max(1, edge_idx.shape[1] // 500)
for j in range(0, edge_idx.shape[1], step):
    src, dst = edge_idx[0, j], edge_idx[1, j]
    pts = pos[[src, dst]]
    ax2.plot(pts[:, 0], pts[:, 1], pts[:, 2],
             'b-', alpha=0.15, linewidth=0.3)
ax2.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c='red', s=1, alpha=0.5)
ax2.set_title(f'Edges ({edge_idx.shape[1]})', fontsize=10)
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

# --- Plot 3: Edge length distribution ---
ax3 = fig.add_subplot(133)
edge_lengths = sample.edge_attr.numpy().flatten()
ax3.hist(edge_lengths, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax3.set_title('Edge Length Distribution', fontsize=10)
ax3.set_xlabel('Normalized Distance')
ax3.set_ylabel('Count')

fig.suptitle(f'Graph: {sample.item_name}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Phase 2 — GNN Training

In [ ]:
# 2.1  Train / Validation split + DataLoaders
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

# Stratified split on labels
labels = [int(d.y.item()) for d in graph_list]
train_data, val_data = train_test_split(
    graph_list, test_size=VAL_SPLIT,
    stratify=labels, random_state=42
)

print(f'📦 Train: {len(train_data)} graphs')
print(f'📦 Val  : {len(val_data)} graphs')

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)

# Quick sanity check
batch = next(iter(train_loader))
print(f'\n🔍 Sample batch:')
print(f'   Batch x     : {batch.x.shape}')
print(f'   Batch edges  : {batch.edge_index.shape}')
print(f'   Batch labels : {batch.y.shape}')
print(f'   Batch vector : {batch.batch.shape} (max={batch.batch.max().item()})')

In [ ]:
# 2.2  Define GNN model, optimizer, loss functions
import torch.nn as nn
from utils.gnn_model import StressGNN

model = StressGNN(
    in_channels=IN_CHANNELS,
    hidden_channels=HIDDEN_CHANNELS,
    num_gnn_layers=NUM_GNN_LAYERS,
    edge_dim=EDGE_DIM,
    dropout=DROPOUT,
).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'🧠 StressGNN Model')
print(f'   Total params    : {total_params:,}')
print(f'   Trainable params: {trainable_params:,}')
print(f'   GNN layers      : {NUM_GNN_LAYERS}')
print(f'   Hidden dim      : {HIDDEN_CHANNELS}')
print()
print(model)

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Learning rate scheduler (reduce on plateau)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, verbose=True
)

# Loss functions
stress_loss_fn = nn.MSELoss()          # Regression: predict max stress
risk_loss_fn   = nn.BCELoss()          # Per-node: risk classification

print(f'\n⚙️  Optimizer: Adam (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})')
print(f'   Loss = {LAMBDA_STRESS}×MSE(stress) + {LAMBDA_RISK}×BCE(node_risk)')

In [ ]:
# 2.3  Training loop
import time

history = {
    'train_loss': [], 'val_loss': [],
    'train_stress_mae': [], 'val_stress_mae': [],
}

best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

# Save best model to Google Drive
BEST_MODEL_PATH = os.path.join(DRIVE_MODEL_DIR, 'best_gnn_model.pth')

print(f'🚀 Training for {NUM_EPOCHS} epochs on {DEVICE}...')
print(f'   Early stopping patience: {EARLY_STOP_PATIENCE}')
print(f'   Model checkpoint: {BEST_MODEL_PATH}')
print('=' * 70)

train_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    # ---- TRAIN ----
    model.train()
    train_loss_sum = 0.0
    train_mae_sum = 0.0
    train_count = 0

    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()

        out = model(batch)

        # Stress regression loss
        stress_pred = out['stress']              # (B, 1)
        stress_true = batch.stress_value.view(-1, 1)  # (B, 1)
        loss_stress = stress_loss_fn(stress_pred, stress_true)

        # Node-risk classification loss
        # Use the graph-level label as a soft target for all nodes in that graph
        node_risk = out['node_risk']             # (N_total, 1)
        node_labels = batch.y[batch.batch].view(-1, 1)  # broadcast graph label to nodes
        loss_risk = risk_loss_fn(node_risk, node_labels)

        # Combined loss
        loss = LAMBDA_STRESS * loss_stress + LAMBDA_RISK * loss_risk
        loss.backward()
        optimizer.step()

        bs = stress_true.shape[0]
        train_loss_sum += loss.item() * bs
        train_mae_sum += (stress_pred - stress_true).abs().sum().item()
        train_count += bs

    train_loss = train_loss_sum / train_count
    train_mae = train_mae_sum / train_count

    # ---- VALIDATE ----
    model.eval()
    val_loss_sum = 0.0
    val_mae_sum = 0.0
    val_count = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            out = model(batch)

            stress_pred = out['stress']
            stress_true = batch.stress_value.view(-1, 1)
            loss_stress = stress_loss_fn(stress_pred, stress_true)

            node_risk = out['node_risk']
            node_labels = batch.y[batch.batch].view(-1, 1)
            loss_risk = risk_loss_fn(node_risk, node_labels)

            loss = LAMBDA_STRESS * loss_stress + LAMBDA_RISK * loss_risk

            bs = stress_true.shape[0]
            val_loss_sum += loss.item() * bs
            val_mae_sum += (stress_pred - stress_true).abs().sum().item()
            val_count += bs

    val_loss = val_loss_sum / val_count
    val_mae = val_mae_sum / val_count

    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_stress_mae'].append(train_mae)
    history['val_stress_mae'].append(val_mae)

    # LR scheduler step
    scheduler.step(val_loss)

    # Best model checkpoint (saved to Google Drive)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    else:
        patience_counter += 1

    # Print progress
    if epoch % 5 == 0 or epoch == 1:
        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} │ '
              f'Train Loss: {train_loss:.4f}  MAE: {train_mae:.2f} │ '
              f'Val Loss: {val_loss:.4f}  MAE: {val_mae:.2f} │ '
              f'LR: {lr:.1e} │ '
              f'Best: ep{best_epoch}')

    # Early stopping
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f'\n⏹️  Early stopping at epoch {epoch} (no improvement for {EARLY_STOP_PATIENCE} epochs)')
        break

train_time = time.time() - train_start
print(f'\n✅ Training complete in {train_time/60:.1f} min')
print(f'   Best val loss: {best_val_loss:.4f} at epoch {best_epoch}')
print(f'   Model saved to: {BEST_MODEL_PATH}')

In [ ]:
# 2.4  Training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss', color='#2196F3')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='#F44336')
axes[0].axvline(best_epoch - 1, color='green', linestyle='--', alpha=0.5, label=f'Best (ep {best_epoch})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Combined Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# MAE curves
axes[1].plot(history['train_stress_mae'], label='Train MAE', color='#2196F3')
axes[1].plot(history['val_stress_mae'],   label='Val MAE',   color='#F44336')
axes[1].axvline(best_epoch - 1, color='green', linestyle='--', alpha=0.5, label=f'Best (ep {best_epoch})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Stress MAE (MPa)')
axes[1].set_title('Stress Prediction MAE')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('GNN Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Phase 3 — Evaluation & Inference

In [ ]:
# 3.1  Evaluate on validation set
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score

# Load best model from Drive
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()

all_stress_pred = []
all_stress_true = []
all_label_pred  = []
all_label_true  = []

with torch.no_grad():
    for batch in val_loader:
        batch = batch.to(DEVICE)
        out = model(batch)

        # Stress regression
        stress_pred = out['stress'].cpu().numpy().flatten()
        stress_true = batch.stress_value.cpu().numpy().flatten()
        all_stress_pred.extend(stress_pred)
        all_stress_true.extend(stress_true)

        # Classification (threshold: if predicted stress >= threshold → FAIL)
        label_true = batch.y.cpu().numpy().flatten()
        label_pred = (stress_pred >= threshold).astype(float)
        all_label_pred.extend(label_pred)
        all_label_true.extend(label_true)

all_stress_pred = np.array(all_stress_pred)
all_stress_true = np.array(all_stress_true)

# Metrics
mae = mean_absolute_error(all_stress_true, all_stress_pred)
r2  = r2_score(all_stress_true, all_stress_pred)
acc = accuracy_score(all_label_true, all_label_pred)

print('=' * 50)
print('📊  Validation Results (Best Model)')
print('=' * 50)
print(f'  Stress columns used : {len(STRESS_COLS)}')
print(f'  Stress MAE          : {mae:.2f} MPa')
print(f'  Stress R²           : {r2:.4f}')
print(f'  Classification      : {acc*100:.1f}% accuracy')
print(f'  Samples             : {len(all_stress_true)}')
print('=' * 50)

# Scatter plot: predicted vs true stress
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(all_stress_true, all_stress_pred, alpha=0.6, c='steelblue', s=40)
lims = [min(all_stress_true.min(), all_stress_pred.min()),
        max(all_stress_true.max(), all_stress_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('True Stress (MPa)', fontsize=12)
ax.set_ylabel('Predicted Stress (MPa)', fontsize=12)
ax.set_title(f'GNN Stress Prediction (R² = {r2:.3f})', fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 3.2  Single-file inference
from utils.mesh_to_graph import mesh_to_graph
from utils.gnn_model import predict_stress

def analyze_stl(stl_path: str):
    """
    Run the full V2 inference pipeline on a single STL file.

    Returns dict with predicted stress, risk score, and risk location.
    """
    print(f'\n🔍 Analyzing: {os.path.basename(stl_path)}')
    print('   Step 1: Converting mesh to graph...')
    data = mesh_to_graph(stl_path, target_faces=TARGET_FACES)
    print(f'   → {data.num_nodes} nodes, {data.edge_index.shape[1]} edges')

    print('   Step 2: Running GNN inference...')
    result = predict_stress(model, data, device=DEVICE)

    stress = result['predicted_stress']
    risk = result['max_risk_score']
    risk_pos = result['max_risk_position']
    status = '🔴 FAIL' if stress >= threshold else '🟢 PASS'

    print(f'\n   ═══════════════════════════════════════')
    print(f'   Result: {status}')
    print(f'   Predicted Max Stress : {stress:.2f} MPa')
    print(f'   Fail Threshold       : {threshold:.2f} MPa')
    print(f'   Max Risk Score       : {risk:.4f}')
    print(f'   Risk Zone Center     : ({risk_pos[0]:.2f}, {risk_pos[1]:.2f}, {risk_pos[2]:.2f})')
    print(f'   ═══════════════════════════════════════')

    return result, data

# --- Run on a validation sample ---
sample_path = stl_files[0]  # Change index or use your own STL file
result, data = analyze_stl(sample_path)

In [ ]:
# 3.3  Visualization — 3D risk heatmap + danger zone bounding box
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def visualize_risk(
    data,
    result,
    stl_name: str = 'Part',
    bbox_radius: float = None,
):
    """
    3D scatter plot of the mesh colored by per-node risk score,
    with a translucent red bounding box around the danger zone.
    """
    # Get per-node risk scores
    model.eval()
    data_device = data.to(DEVICE)
    with torch.no_grad():
        out = model(data_device)
    node_risk = out['node_risk'].cpu().numpy().flatten()

    pos = data.pos.numpy()
    risk_center = result['max_risk_position']

    # Auto-compute bbox radius if not specified
    if bbox_radius is None:
        extents = pos.max(axis=0) - pos.min(axis=0)
        bbox_radius = extents.max() * 0.08  # 8% of largest extent

    fig = plt.figure(figsize=(16, 7))

    # --- Plot 1: Full model with risk heatmap ---
    ax1 = fig.add_subplot(121, projection='3d')
    sc = ax1.scatter(
        pos[:, 0], pos[:, 1], pos[:, 2],
        c=node_risk, cmap='YlOrRd', s=5, alpha=0.8,
        vmin=0, vmax=1,
    )
    plt.colorbar(sc, ax=ax1, shrink=0.6, label='Risk Score')
    ax1.set_title(f'{stl_name} — Node Risk Heatmap', fontsize=11)
    ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

    # --- Plot 2: Danger zone close-up with bounding box ---
    ax2 = fig.add_subplot(122, projection='3d')

    # Color points by risk
    sc2 = ax2.scatter(
        pos[:, 0], pos[:, 1], pos[:, 2],
        c=node_risk, cmap='YlOrRd', s=5, alpha=0.6,
        vmin=0, vmax=1,
    )

    # Draw translucent red bounding box around danger zone
    r = bbox_radius
    cx, cy, cz = risk_center

    # 8 corners of the bounding box
    corners = np.array([
        [cx-r, cy-r, cz-r], [cx+r, cy-r, cz-r],
        [cx+r, cy+r, cz-r], [cx-r, cy+r, cz-r],
        [cx-r, cy-r, cz+r], [cx+r, cy-r, cz+r],
        [cx+r, cy+r, cz+r], [cx-r, cy+r, cz+r],
    ])

    # 6 faces of the box
    faces = [
        [corners[j] for j in [0, 1, 2, 3]],
        [corners[j] for j in [4, 5, 6, 7]],
        [corners[j] for j in [0, 1, 5, 4]],
        [corners[j] for j in [2, 3, 7, 6]],
        [corners[j] for j in [1, 2, 6, 5]],
        [corners[j] for j in [0, 3, 7, 4]],
    ]

    box = Poly3DCollection(faces, alpha=0.15, facecolor='red', edgecolor='red', linewidth=1.5)
    ax2.add_collection3d(box)

    # Mark the risk center
    ax2.scatter(*risk_center, c='red', s=100, marker='X', zorder=10,
                label=f'Risk Center ({cx:.1f}, {cy:.1f}, {cz:.1f})')

    # Zoom in around the danger zone
    zoom = bbox_radius * 5
    ax2.set_xlim(cx - zoom, cx + zoom)
    ax2.set_ylim(cy - zoom, cy + zoom)
    ax2.set_zlim(cz - zoom, cz + zoom)

    stress_val = result['predicted_stress']
    ax2.set_title(
        f'Danger Zone — {stress_val:.1f} MPa\n'
        f'({"FAIL" if stress_val >= threshold else "PASS"})',
        fontsize=11, color='red' if stress_val >= threshold else 'green'
    )
    ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
    ax2.legend(fontsize=9)

    plt.suptitle(
        f'🧠 GNN Structural Analysis — {stl_name}',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

# --- Visualize the sample ---
visualize_risk(data, result, stl_name=os.path.basename(sample_path))

In [ ]:
# 3.4  Upload your own STL file for analysis
try:
    from google.colab import files
    print('📂 Upload an STL file for analysis:')
    uploaded = files.upload()

    for filename in uploaded:
        upload_path = f'/content/{filename}'
        with open(upload_path, 'wb') as f:
            f.write(uploaded[filename])

        result, data = analyze_stl(upload_path)
        visualize_risk(data, result, stl_name=filename)

except ImportError:
    print('Not running in Colab. To analyze a local file:')
    print('  result, data = analyze_stl("/path/to/your/file.stl")')
    print('  visualize_risk(data, result, stl_name="your_file.stl")')

In [ ]:
# 3.5  Save final model + config to Google Drive
import json

# Save model weights
final_model_path = os.path.join(DRIVE_MODEL_DIR, 'stress_gnn_v2_final.pth')
torch.save(model.state_dict(), final_model_path)
print(f'💾 Model saved to Drive: {final_model_path}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Save model config for reproducibility
config = {
    'model': 'StressGNN',
    'version': 'v2',
    'in_channels': IN_CHANNELS,
    'hidden_channels': HIDDEN_CHANNELS,
    'num_gnn_layers': NUM_GNN_LAYERS,
    'edge_dim': EDGE_DIM,
    'target_faces': TARGET_FACES,
    'stress_columns_used': STRESS_COLS,
    'num_stress_columns': len(STRESS_COLS),
    'stress_threshold_mpa': float(threshold),
    'best_epoch': best_epoch,
    'best_val_loss': float(best_val_loss),
}
config_path = os.path.join(DRIVE_MODEL_DIR, 'gnn_model_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'📄 Config saved to Drive: {config_path}')
print(json.dumps(config, indent=2))

## Phase 4 — Presentation Dashboard

Polished, **presentation-ready** visualizations that make the project’s 
value immediately obvious — ideal for portfolio, reports, or demos.

These cells use the **trained model** from Phase 2 and the **raw data/labels** 
from Phase 1. Just run all cells linearly (Phases 0–4).

In [ ]:
# 4.1  Dataset Overview Dashboard
#      Stress distribution + class balance + per-load-case ranges
#
#      Uses: df, threshold, STRESS_COLS  (from Phase 1)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# --- Premium dark theme ---
plt.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor':   '#161B22',
    'axes.edgecolor':   '#30363D',
    'axes.labelcolor':  '#C9D1D9',
    'text.color':       '#C9D1D9',
    'xtick.color':      '#8B949E',
    'ytick.color':      '#8B949E',
    'grid.color':       '#21262D',
    'grid.alpha':       0.6,
    'font.family':      'sans-serif',
    'font.size':        10,
})

# Color palette (reused across Phase 4)
PASS_COLOR = '#3FB950'
FAIL_COLOR = '#F85149'
ACCENT     = '#58A6FF'
GOLD       = '#D29922'

fig = plt.figure(figsize=(20, 6))
gs = gridspec.GridSpec(1, 3, width_ratios=[2, 1, 2], wspace=0.35)

# --- Panel 1: Stress Distribution with PASS/FAIL shading ---
ax1 = fig.add_subplot(gs[0])
stress_vals = df['max_stress_all'].values
bins = np.linspace(stress_vals.min(), stress_vals.max(), 40)

ax1.hist(stress_vals[stress_vals < threshold], bins=bins,
         color=PASS_COLOR, alpha=0.85, label='PASS', edgecolor='none')
ax1.hist(stress_vals[stress_vals >= threshold], bins=bins,
         color=FAIL_COLOR, alpha=0.85, label='FAIL', edgecolor='none')
ax1.axvline(threshold, color=GOLD, linewidth=2.5, linestyle='--',
            label=f'Threshold: {threshold:.1f} MPa', zorder=5)

ax1.set_xlabel('Max Stress (MPa)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Parts', fontsize=12, fontweight='bold')
ax1.set_title('Stress Distribution', fontsize=14, fontweight='bold', pad=12)
ax1.legend(fontsize=10, loc='upper right',
           facecolor='#161B22', edgecolor='#30363D')
ax1.grid(axis='y')

# --- Panel 2: Class Balance Donut Chart ---
ax2 = fig.add_subplot(gs[1])
n_pass = int(len(df) - df['label'].sum())
n_fail = int(df['label'].sum())
wedges, texts, autotexts = ax2.pie(
    [n_pass, n_fail],
    labels=['PASS', 'FAIL'],
    colors=[PASS_COLOR, FAIL_COLOR],
    autopct='%1.0f%%',
    startangle=90,
    pctdistance=0.78,
    wedgeprops=dict(width=0.45, edgecolor='#0D1117', linewidth=2),
    textprops=dict(fontsize=12, fontweight='bold'),
)
for at in autotexts:
    at.set_color('#FFFFFF')
    at.set_fontweight('bold')
ax2.set_title('Class Balance', fontsize=14, fontweight='bold', pad=12)
ax2.text(0, 0, f'{len(df)}\nparts', ha='center', va='center',
         fontsize=16, fontweight='bold', color='#C9D1D9')

# --- Panel 3: Per-Load-Case Box Plot ---
ax3 = fig.add_subplot(gs[2])
box_data = [df[col].values for col in STRESS_COLS]
short_names = [c.replace('stress_', '').replace('_mpa', '')
               .replace('_', ' ').title() for c in STRESS_COLS]

bp = ax3.boxplot(box_data, labels=short_names, patch_artist=True,
                 medianprops=dict(color=GOLD, linewidth=2),
                 whiskerprops=dict(color='#8B949E'),
                 capprops=dict(color='#8B949E'),
                 flierprops=dict(marker='o', markerfacecolor=FAIL_COLOR,
                                 markersize=4, alpha=0.6))

cmap = plt.cm.get_cmap('cool', len(STRESS_COLS))
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(cmap(i))
    patch.set_alpha(0.75)
    patch.set_edgecolor('#30363D')

ax3.axhline(threshold, color=FAIL_COLOR, linewidth=1.5, linestyle=':',
            alpha=0.7, label='Fail Threshold')
ax3.set_ylabel('Stress (MPa)', fontsize=12, fontweight='bold')
ax3.set_title('Stress by Load Case', fontsize=14, fontweight='bold', pad=12)
ax3.tick_params(axis='x', rotation=25)
ax3.legend(fontsize=9, facecolor='#161B22', edgecolor='#30363D')
ax3.grid(axis='y')

fig.suptitle('📊  Dataset Overview — AI 3D Stress Validator V2',
             fontsize=18, fontweight='bold', y=1.02, color='#F0F6FC')
plt.tight_layout()
plt.show()

In [ ]:
# 4.2  Model Performance Report Card
#      Confusion matrix + prediction scatter + residual distribution
#
#      Uses: all_stress_true/pred, all_label_true/pred  (from Phase 3)
#             PASS_COLOR, FAIL_COLOR, ACCENT, GOLD     (from cell 4.1)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from sklearn.metrics import confusion_matrix, r2_score, mean_absolute_error

fig = plt.figure(figsize=(20, 7))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1.4, 1], wspace=0.3)

# Compute metrics
_mae = mean_absolute_error(all_stress_true, all_stress_pred)
_r2  = r2_score(all_stress_true, all_stress_pred)

# --- Panel 1: Confusion Matrix ---
ax1 = fig.add_subplot(gs[0])
cm = confusion_matrix(all_label_true, all_label_pred)
cm_labels = np.array([['True\nPASS', 'False\nFAIL'],
                      ['False\nPASS', 'True\nFAIL']])

im = ax1.imshow(cm, cmap='RdYlGn_r', aspect='auto', alpha=0.85)
for i in range(2):
    for j in range(2):
        label_text = f'{cm_labels[i,j]}\n\n{cm[i,j]}'
        ax1.text(j, i, label_text, ha='center', va='center',
                 fontsize=13, fontweight='bold',
                 color='white' if cm[i,j] > cm.max()/2 else '#C9D1D9')

ax1.set_xticks([0, 1])
ax1.set_yticks([0, 1])
ax1.set_xticklabels(['PASS', 'FAIL'], fontsize=12, fontweight='bold')
ax1.set_yticklabels(['PASS', 'FAIL'], fontsize=12, fontweight='bold')
ax1.set_xlabel('Predicted', fontsize=13, fontweight='bold')
ax1.set_ylabel('Actual', fontsize=13, fontweight='bold')
ax1.set_title('Confusion Matrix', fontsize=14, fontweight='bold', pad=12)

_acc = np.trace(cm) / cm.sum()
ax1.text(0.5, -0.18, f'Accuracy: {_acc*100:.1f}%',
         transform=ax1.transAxes, ha='center', fontsize=13,
         fontweight='bold', color=PASS_COLOR if _acc > 0.8 else GOLD)

# --- Panel 2: Predicted vs True Stress ---
ax2 = fig.add_subplot(gs[1])

lims = [min(all_stress_true.min(), all_stress_pred.min()) * 0.9,
        max(all_stress_true.max(), all_stress_pred.max()) * 1.1]
band_x = np.linspace(lims[0], lims[1], 100)
ax2.fill_between(band_x, band_x * 0.8, band_x * 1.2,
                 alpha=0.08, color=ACCENT, label='±20% band')
ax2.fill_between(band_x, band_x * 0.9, band_x * 1.1,
                 alpha=0.12, color=ACCENT, label='±10% band')

errors = np.abs(all_stress_pred - all_stress_true)
sc = ax2.scatter(all_stress_true, all_stress_pred,
                 c=errors, cmap='YlOrRd', s=60, alpha=0.85,
                 edgecolors='#30363D', linewidth=0.5, zorder=5)
plt.colorbar(sc, ax=ax2, shrink=0.8, label='Abs Error (MPa)')

ax2.plot(lims, lims, '--', color=PASS_COLOR, linewidth=2,
         label='Perfect', zorder=4)

ax2.set_xlim(lims)
ax2.set_ylim(lims)
ax2.set_xlabel('True Stress (MPa)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Predicted Stress (MPa)', fontsize=12, fontweight='bold')
ax2.set_title(f'Prediction Accuracy   R² = {_r2:.3f}',
              fontsize=14, fontweight='bold', pad=12)
ax2.legend(fontsize=9, loc='upper left',
           facecolor='#161B22', edgecolor='#30363D')
ax2.grid(alpha=0.3)
ax2.set_aspect('equal')

# --- Panel 3: Residual Distribution ---
ax3 = fig.add_subplot(gs[2])
residuals = all_stress_pred - all_stress_true
ax3.hist(residuals, bins=25, color=ACCENT, alpha=0.8,
         edgecolor='#0D1117', linewidth=0.8)
ax3.axvline(0, color=PASS_COLOR, linewidth=2, linestyle='--',
            label='Zero Error')
ax3.axvline(residuals.mean(), color=GOLD, linewidth=2,
            label=f'Mean: {residuals.mean():.2f}')

ax3.set_xlabel('Prediction Error (MPa)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Count', fontsize=12, fontweight='bold')
ax3.set_title('Error Distribution', fontsize=14, fontweight='bold', pad=12)
ax3.legend(fontsize=9, facecolor='#161B22', edgecolor='#30363D')
ax3.grid(axis='y')

stats_text = (f'MAE  : {_mae:.2f} MPa\n'
              f'R²   : {_r2:.4f}\n'
              f'Acc  : {_acc*100:.1f}%\n'
              f'Bias : {residuals.mean():+.2f}')
ax3.text(0.97, 0.97, stats_text, transform=ax3.transAxes,
         va='top', ha='right', fontsize=10, family='monospace',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#21262D',
                   edgecolor='#30363D', alpha=0.9))

fig.suptitle('🧠  GNN Model Performance — Report Card',
             fontsize=18, fontweight='bold', y=1.02, color='#F0F6FC')
plt.tight_layout()
plt.show()

In [ ]:
# 4.3  3D Mesh Comparison — PASS vs FAIL
#      Side-by-side risk heatmap on actual mesh geometry
#
#      Uses: val_data, model, DEVICE  (from Phases 2-3)

import matplotlib.pyplot as plt
import numpy as np
import torch

# Find one PASS and one FAIL sample from the validation set
pass_sample = None
fail_sample = None
for d in val_data:
    if d.y.item() == 0 and pass_sample is None:
        pass_sample = d
    elif d.y.item() == 1 and fail_sample is None:
        fail_sample = d
    if pass_sample is not None and fail_sample is not None:
        break

if pass_sample is None:
    pass_sample = val_data[0]
if fail_sample is None:
    fail_sample = val_data[-1]

fig = plt.figure(figsize=(20, 8))

model.eval()
for idx, (sample, title_tag) in enumerate([
    (pass_sample, 'PASS'), (fail_sample, 'FAIL')
]):
    with torch.no_grad():
        out = model(sample.to(DEVICE))
    node_risk = out['node_risk'].cpu().numpy().flatten()
    stress_pred = out['stress'].cpu().item()
    pos = sample.pos.numpy()

    # --- Full view ---
    ax = fig.add_subplot(1, 4, idx * 2 + 1, projection='3d')
    sc = ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2],
                    c=node_risk, cmap='inferno', s=6, alpha=0.85,
                    vmin=0, vmax=1)
    verdict_color = PASS_COLOR if title_tag == 'PASS' else FAIL_COLOR
    ax.set_title(f'{title_tag}  •  {stress_pred:.1f} MPa\n'
                 f'{sample.item_name}',
                 fontsize=13, fontweight='bold', color=verdict_color, pad=10)
    ax.set_xlabel('X', fontsize=8)
    ax.set_ylabel('Y', fontsize=8)
    ax.set_zlabel('Z', fontsize=8)
    ax.tick_params(labelsize=7)

    # --- Zoomed danger zone ---
    ax_z = fig.add_subplot(1, 4, idx * 2 + 2, projection='3d')
    top_k = max(1, int(len(node_risk) * 0.15))
    high_risk_idx = np.argsort(node_risk)[-top_k:]
    hr_pos = pos[high_risk_idx]
    hr_risk = node_risk[high_risk_idx]

    sc2 = ax_z.scatter(hr_pos[:, 0], hr_pos[:, 1], hr_pos[:, 2],
                       c=hr_risk, cmap='hot', s=18, alpha=0.95,
                       vmin=0, vmax=1, edgecolors='#30363D', linewidth=0.3)

    ax_z.scatter(pos[:, 0], pos[:, 1], pos[:, 2],
                c='#21262D', s=1, alpha=0.2)

    risk_center = hr_pos.mean(axis=0)
    ax_z.scatter(*risk_center, c='cyan', s=120, marker='X',
                zorder=10, edgecolors='white', linewidth=1)

    ax_z.set_title(f'Danger Zone  •  Top 15% Risk\n'
                   f'Max Risk: {node_risk.max():.3f}',
                   fontsize=11, fontweight='bold', color=GOLD, pad=10)
    ax_z.set_xlabel('X', fontsize=8)
    ax_z.set_ylabel('Y', fontsize=8)
    ax_z.set_zlabel('Z', fontsize=8)
    ax_z.tick_params(labelsize=7)

cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.65])
cbar = fig.colorbar(sc, cax=cbar_ax)
cbar.set_label('Node Risk Score', fontsize=11, fontweight='bold')

fig.suptitle('🔬  3D Structural Comparison — PASS vs FAIL Part',
             fontsize=18, fontweight='bold', y=1.01, color='#F0F6FC')
plt.tight_layout(rect=[0, 0, 0.92, 0.96])
plt.show()

In [ ]:
# 4.4  Single-Part Failure Analysis — 3D Object with Failure Point
#      Shows one part with its predicted failure location clearly marked.
#
#      Uses: model, DEVICE, stl_files, threshold  (from Phases 1-3)
#             mesh_to_graph, predict_stress          (from utils)

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import numpy as np
import torch
from utils.mesh_to_graph import mesh_to_graph
from utils.gnn_model import predict_stress

# --- Pick the highest-stress part from validation set for impact ---
stress_values = []
model.eval()
for d in val_data:
    with torch.no_grad():
        out = model(d.to(DEVICE))
    stress_values.append(out['stress'].cpu().item())

worst_idx = np.argmax(stress_values)
target = val_data[worst_idx]

# Run full inference
with torch.no_grad():
    out = model(target.to(DEVICE))
node_risk = out['node_risk'].cpu().numpy().flatten()
pred_stress = out['stress'].cpu().item()
pos = target.pos.numpy()

# Find failure point (highest-risk node)
fail_node = np.argmax(node_risk)
fail_pos = pos[fail_node]
fail_risk = node_risk[fail_node]

# Top 5% risk zone
top_k = max(1, int(len(node_risk) * 0.05))
danger_idx = np.argsort(node_risk)[-top_k:]
danger_pos = pos[danger_idx]
danger_center = danger_pos.mean(axis=0)

# Bounding box around danger zone
bbox_margin = (pos.max(axis=0) - pos.min(axis=0)).max() * 0.06
bbox_min = danger_pos.min(axis=0) - bbox_margin
bbox_max = danger_pos.max(axis=0) + bbox_margin

verdict = 'FAIL' if pred_stress >= threshold else 'PASS'
verdict_color = FAIL_COLOR if verdict == 'FAIL' else PASS_COLOR

# ---- FIGURE ----
fig = plt.figure(figsize=(22, 9))

# --- Panel 1: Full 3D model with risk heatmap ---
ax1 = fig.add_subplot(131, projection='3d')
sc1 = ax1.scatter(pos[:, 0], pos[:, 1], pos[:, 2],
                   c=node_risk, cmap='YlOrRd', s=8, alpha=0.85,
                   vmin=0, vmax=1)

# Mark failure point with large marker
ax1.scatter(*fail_pos, c='red', s=200, marker='*', zorder=10,
            edgecolors='white', linewidth=1.5,
            label=f'Failure Point (risk={fail_risk:.3f})')

plt.colorbar(sc1, ax=ax1, shrink=0.6, pad=0.08, label='Risk Score')
ax1.set_title(f'Full Model — Risk Heatmap\n'
              f'{target.item_name}',
              fontsize=13, fontweight='bold', pad=12)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.legend(fontsize=9, loc='upper left',
           facecolor='#161B22', edgecolor='#30363D')

# --- Panel 2: Zoomed failure zone with bounding box ---
ax2 = fig.add_subplot(132, projection='3d')

# Full shape ghost
ax2.scatter(pos[:, 0], pos[:, 1], pos[:, 2],
            c='#30363D', s=2, alpha=0.15)

# Danger nodes
sc2 = ax2.scatter(danger_pos[:, 0], danger_pos[:, 1], danger_pos[:, 2],
                   c=node_risk[danger_idx], cmap='hot', s=25, alpha=0.95,
                   vmin=0, vmax=1, edgecolors='#30363D', linewidth=0.3)

# Failure point marker
ax2.scatter(*fail_pos, c='cyan', s=250, marker='X', zorder=10,
            edgecolors='white', linewidth=2,
            label=f'Failure: ({fail_pos[0]:.1f}, {fail_pos[1]:.1f}, {fail_pos[2]:.1f})')

# Draw translucent red bounding box
bmin, bmax = bbox_min, bbox_max
corners = np.array([
    [bmin[0], bmin[1], bmin[2]], [bmax[0], bmin[1], bmin[2]],
    [bmax[0], bmax[1], bmin[2]], [bmin[0], bmax[1], bmin[2]],
    [bmin[0], bmin[1], bmax[2]], [bmax[0], bmin[1], bmax[2]],
    [bmax[0], bmax[1], bmax[2]], [bmin[0], bmax[1], bmax[2]],
])
faces = [
    [corners[j] for j in [0,1,2,3]], [corners[j] for j in [4,5,6,7]],
    [corners[j] for j in [0,1,5,4]], [corners[j] for j in [2,3,7,6]],
    [corners[j] for j in [1,2,6,5]], [corners[j] for j in [0,3,7,4]],
]
box = Poly3DCollection(faces, alpha=0.12, facecolor='red',
                       edgecolor='red', linewidth=1.5)
ax2.add_collection3d(box)

# Zoom to danger zone
zoom_pad = (bmax - bmin).max() * 2
ax2.set_xlim(danger_center[0] - zoom_pad, danger_center[0] + zoom_pad)
ax2.set_ylim(danger_center[1] - zoom_pad, danger_center[1] + zoom_pad)
ax2.set_zlim(danger_center[2] - zoom_pad, danger_center[2] + zoom_pad)

ax2.set_title(f'Failure Zone Close-Up\n'
              f'Top 5% highest-risk nodes',
              fontsize=13, fontweight='bold', color=FAIL_COLOR, pad=12)
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
ax2.legend(fontsize=9, loc='upper left',
           facecolor='#161B22', edgecolor='#30363D')

# --- Panel 3: Verdict Card ---
ax3 = fig.add_subplot(133)
ax3.axis('off')

# Big verdict text
ax3.text(0.5, 0.85, verdict, fontsize=60, fontweight='bold',
         ha='center', va='center', color=verdict_color,
         transform=ax3.transAxes)

# Stats block
info_lines = [
    f'Part: {target.item_name}',
    f'',
    f'Predicted Stress:  {pred_stress:.2f} MPa',
    f'Fail Threshold:    {threshold:.2f} MPa',
    f'',
    f'Max Risk Score:    {fail_risk:.4f}',
    f'Failure Node:      #{fail_node}',
    f'',
    f'Failure Location:',
    f'  X = {fail_pos[0]:.3f}',
    f'  Y = {fail_pos[1]:.3f}',
    f'  Z = {fail_pos[2]:.3f}',
    f'',
    f'Danger Zone:  {top_k} nodes ({top_k/len(node_risk)*100:.1f}%)',
]
info_text = '\n'.join(info_lines)
ax3.text(0.5, 0.42, info_text, fontsize=12, family='monospace',
         ha='center', va='center', transform=ax3.transAxes,
         bbox=dict(boxstyle='round,pad=0.8', facecolor='#21262D',
                   edgecolor='#30363D', alpha=0.95),
         linespacing=1.6)

fig.suptitle('🚨  Single-Part Failure Analysis — Highest Stress Part',
             fontsize=18, fontweight='bold', y=1.01, color='#F0F6FC')
plt.tight_layout()
plt.show()

print(f'\n{"="*60}')
print(f'  FAILURE POINT COORDINATES')
print(f'  Part: {target.item_name}')
print(f'  Location:  X={fail_pos[0]:.4f}  Y={fail_pos[1]:.4f}  Z={fail_pos[2]:.4f}')
print(f'  Risk Score: {fail_risk:.4f}')
print(f'  Predicted Stress: {pred_stress:.2f} MPa')
print(f'  Verdict: {verdict}')
print(f'{"="*60}')

In [ ]:
# 4.5  Pipeline Summary — Final Report
#
#      Uses: graph_list, model, _r2, _mae  (from Phases 1-4)
#             NUM_GNN_LAYERS, HIDDEN_CHANNELS, STRESS_COLS, best_epoch  (from Phase 0)

from IPython.display import display, HTML
import numpy as np

n_parts = len(graph_list)
avg_nodes = int(np.mean([g.num_nodes for g in graph_list]))
avg_edges = int(np.mean([g.edge_index.shape[1] for g in graph_list]))
total_params = sum(p.numel() for p in model.parameters())

html = f"""
<div style="
    background: linear-gradient(135deg, #0D1117 0%, #161B22 50%, #1A1E2E 100%);
    border: 1px solid #30363D;
    border-radius: 16px;
    padding: 32px 40px;
    font-family: 'Segoe UI', system-ui, -apple-system, sans-serif;
    color: #C9D1D9;
    max-width: 820px;
    margin: 20px auto;
    box-shadow: 0 8px 32px rgba(0,0,0,0.4);
">

  <h2 style="text-align:center; color:#F0F6FC; margin:0 0 6px 0;
             font-size:24px; letter-spacing:0.5px;">
    🧠 AI 3D Stress Validator — V2 GNN Summary
  </h2>
  <p style="text-align:center; color:#8B949E; margin:0 0 28px 0; font-size:13px;">
    Physics-Informed Structural Validation via Graph Neural Networks
  </p>

  <div style="display:flex; align-items:center; justify-content:center;
              gap:8px; margin:0 0 28px 0; flex-wrap:wrap;">
    <span style="background:#21262D; padding:10px 18px; border-radius:10px;
                 border:1px solid #30363D; font-weight:600; font-size:14px;">📐 STL Mesh</span>
    <span style="color:#58A6FF; font-size:22px;">→</span>
    <span style="background:#21262D; padding:10px 18px; border-radius:10px;
                 border:1px solid #30363D; font-weight:600; font-size:14px;">🕸️ Graph (V,E)</span>
    <span style="color:#58A6FF; font-size:22px;">→</span>
    <span style="background:#21262D; padding:10px 18px; border-radius:10px;
                 border:1px solid #30363D; font-weight:600; font-size:14px;">🧠 GNN</span>
    <span style="color:#58A6FF; font-size:22px;">→</span>
    <span style="background:#21262D; padding:10px 18px; border-radius:10px;
                 border:1px solid #30363D; font-weight:600; font-size:14px;">🎯 Risk Map</span>
  </div>

  <div style="display:grid; grid-template-columns:repeat(4,1fr); gap:14px;
              margin-bottom:24px;">
    <div style="background:#21262D; padding:16px; border-radius:10px;
               border:1px solid #30363D; text-align:center;">
      <div style="font-size:26px; font-weight:700; color:#58A6FF;">{n_parts}</div>
      <div style="font-size:11px; color:#8B949E; margin-top:4px;">Parts Trained</div>
    </div>
    <div style="background:#21262D; padding:16px; border-radius:10px;
               border:1px solid #30363D; text-align:center;">
      <div style="font-size:26px; font-weight:700; color:#D29922;">{total_params:,}</div>
      <div style="font-size:11px; color:#8B949E; margin-top:4px;">Parameters</div>
    </div>
    <div style="background:#21262D; padding:16px; border-radius:10px;
               border:1px solid #30363D; text-align:center;">
      <div style="font-size:26px; font-weight:700; color:#3FB950;">{_r2:.3f}</div>
      <div style="font-size:11px; color:#8B949E; margin-top:4px;">R² Score</div>
    </div>
    <div style="background:#21262D; padding:16px; border-radius:10px;
               border:1px solid #30363D; text-align:center;">
      <div style="font-size:26px; font-weight:700; color:#F85149;">{_mae:.1f}</div>
      <div style="font-size:11px; color:#8B949E; margin-top:4px;">MAE (MPa)</div>
    </div>
  </div>

  <div style="display:grid; grid-template-columns:1fr 1fr; gap:14px;
              margin-bottom:24px;">
    <div style="background:#21262D; padding:16px 20px; border-radius:10px;
               border:1px solid #30363D;">
      <div style="font-size:13px; font-weight:700; color:#58A6FF;
                  margin-bottom:10px;">📐 Graph Stats (avg)</div>
      <div style="font-size:12px; line-height:1.8;">
        Nodes per graph: <b>{avg_nodes:,}</b><br>
        Edges per graph: <b>{avg_edges:,}</b><br>
        Node features: <b>6D</b> (xyz + normals)<br>
        Edge features: <b>1D</b> (distance)
      </div>
    </div>
    <div style="background:#21262D; padding:16px 20px; border-radius:10px;
               border:1px solid #30363D;">
      <div style="font-size:13px; font-weight:700; color:#D29922;
                  margin-bottom:10px;">🧠 Model Config</div>
      <div style="font-size:12px; line-height:1.8;">
        Architecture: <b>StressGNN ({NUM_GNN_LAYERS}-layer GCN)</b><br>
        Hidden dim: <b>{HIDDEN_CHANNELS}</b><br>
        Stress columns: <b>{len(STRESS_COLS)}</b> load cases<br>
        Best epoch: <b>{best_epoch}</b>
      </div>
    </div>
  </div>

  <div style="background: linear-gradient(90deg, #238636 0%, #2EA043 100%);
              padding:14px 24px; border-radius:10px; text-align:center;">
    <span style="font-size:15px; font-weight:700; color:white;
                 letter-spacing:0.5px;">
      ✅  Model trained and validated — ready for inference on new STL files
    </span>
  </div>

</div>
"""

display(HTML(html))